In [418]:
import snowflake.connector
from sqlalchemy import create_engine
from sqlalchemy import text
from snowflake.sqlalchemy import URL
from datetime import datetime, timedelta, timezone,date
import pandas as pd
import numpy as np

In [420]:
df = pd.read_csv('/Users/amanda.huang/Downloads/partner_api_transaction.csv')

In [422]:
pd.set_option('display.max_columns', None)

In [424]:
import json

# Convert JSON string to dict
df_parsed = df['raw_request_json'].apply(json.loads)

# Turn into columns
df_expanded = pd.json_normalize(df_parsed)

# Combine with original df if needed
df_final = pd.concat([df.drop(columns='raw_request_json'), df_expanded], axis=1)

In [426]:
df_final.columns

Index(['id', 'ping_id', 'jluvr', 'id_ad', 'id_brand', 'id_partner',
       'partner_name', 'ping_group', 'raw_response_json', 'accepted',
       'request_datetime', 'response_datetime', 'last_modified', 'income',
       'lastName', 'driverState', 'purpose', 'postalCode', 'jobTitle',
       'bankName', 'bankruptcy', 'mailing', 'directDeposit', 'frequency',
       'driverLicense', 'company', 'military', 'accountLength', 'email',
       'timeJob', 'aff', 'amount', 'creditScore', 'payDateNext', 'homePhone',
       'accountType', 'locality', 'ipAddress', 'userAgent', 'accountNumber',
       'timeAddress', 'accountABA', 'SSN', 'firstName', 'apiToken',
       'termsAgreed', 'mobilePhone', 'streetAddress', 'homeOwner', 'dob',
       'domain', 'minPrice', 'workPhone', 'incomeSource', 'timeAllowed',
       'region', 'payDate'],
      dtype='object')

In [428]:
df_final['dob'] = pd.to_datetime(df_final['dob'], errors='coerce').dt.date

In [448]:
df_final['income'] = pd.to_numeric(df_final['income'], errors='coerce')

In [450]:
df_final['amount'] = pd.to_numeric(df_final['amount'], errors='coerce')

In [452]:
def calculate_age(dob):
    if pd.isna(dob):
        return None
    today = date.today()
    age = today.year - dob.year
    if (today.month, today.day) < (dob.month, dob.day):
        age -= 1
    return age

df_final['age'] = df_final['dob'].apply(calculate_age)


In [454]:
df_final['age'] = df_final['dob'].apply(calculate_age)

In [456]:
def label_age_group(age):
    if age < 30:
        return '30 and under'
    elif 30 < age < 40:
        return '30-40'
    elif 40 < age < 50:
        return '40-50'
    elif 50 < age < 60:
        return '50-60'
    else:
        return '60+'

df_final['age_group'] = df_final['age'].apply(label_age_group)


In [458]:
def label_income_group(income):
    if income < 3000:
        return '3000 and under'
    elif 3000 < income < 4000:
        return '3000-4000'
    elif 4000 < income < 5000:
        return '4000-5000'
    elif 5000 < income < 6000:
        return '5000-6000'
    else:
        return '6000+'

df_final['income_group'] = df_final['income'].apply(label_income_group)

In [460]:
df_final['platform'] = df_final['userAgent'].str.extract(r'\(([^;]+);')

In [462]:
selected_email = ['nikkiavila3740@gmail.com','hippykisses0@gmail.com','gatormomma30@gmail.com','lcmartinezz73@gmail.com','jacsonthf4kp@yahoo.com','hhall1122@yahoo.com','sophiakennedy4@hotmail.com','jordan.rambo@gmail.com','andrea.wilson55@yahoo.com','vkm198@gmail.com']
                  

In [464]:
df_selected_email = df_final[df_final['accepted'] == 1]

In [466]:
df_final = df_final[~df_final['accepted'].isin(selected_email)]

In [468]:
df_selected_email.groupby('age_group').count()/df_selected_email.groupby('age_group').count().sum()

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,amount,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,homeOwner,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,region,payDate,age,income_group,platform
age_group,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
30-40,0.310345,0.310345,0.310345,0.310345,NaN,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345
40-50,0.310345,0.310345,0.310345,0.310345,NaN,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345
50-60,0.155172,0.155172,0.155172,0.155172,NaN,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172
60+,0.224138,0.224138,0.224138,0.224138,NaN,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138,0.224138


In [470]:
df_final.groupby('age_group').count()/df_final.groupby('age_group').count().sum()

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,amount,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,homeOwner,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,region,payDate,age,income_group,platform
age_group,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
30 and under,0.003546,0.003546,0.003546,0.003546,0.0,0.003546,0.003546,0.003546,0.003559,0.003546,0.003546,0.003559,0.003546,0.003559,0.003559,0.003559,0.003546,0.000000,0.003559,0.003559,0.003546,0.003546,0.003546,0.003546,0.003559,0.003559,0.003546,0.003546,0.003559,0.003559,0.003546,0.000000,0.003559,0.000000,0.003559,0.003546,0.003559,0.000000,0.000000,0.003571,0.003546,0.003571,0.003584,0.003559,0.003546,0.003546,0.003559,0.003559,0.003546,0.003559,0.000000,0.003546,0.000000,0.003546,0.003802,0.003559,0.003559,0.003559,0.003546,0.000000
30-40,0.421986,0.421986,0.421986,0.421986,0.4,0.421986,0.421986,0.421986,0.423488,0.421986,0.421986,0.423488,0.421986,0.423488,0.423488,0.423488,0.421986,0.422794,0.423488,0.423488,0.421986,0.421986,0.421986,0.421986,0.423488,0.423488,0.421986,0.421986,0.423488,0.423488,0.421986,0.422383,0.423488,0.424460,0.423488,0.421986,0.423488,0.421245,0.422794,0.421429,0.421986,0.421429,0.419355,0.423488,0.421986,0.421986,0.423488,0.423488,0.421986,0.423488,0.421245,0.421986,0.422794,0.421986,0.406844,0.423488,0.423488,0.423488,0.421986,0.400763
40-50,0.329787,0.329787,0.329787,0.329787,0.0,0.329787,0.329787,0.329787,0.327402,0.329787,0.329787,0.327402,0.329787,0.330961,0.330961,0.330961,0.329787,0.338235,0.330961,0.330961,0.329787,0.329787,0.329787,0.329787,0.330961,0.330961,0.329787,0.329787,0.330961,0.330961,0.329787,0.332130,0.330961,0.330935,0.330961,0.329787,0.330961,0.336996,0.338235,0.332143,0.329787,0.332143,0.333333,0.330961,0.329787,0.329787,0.330961,0.330961,0.329787,0.330961,0.336996,0.329787,0.338235,0.329787,0.334601,0.330961,0.330961,0.330961,0.329787,0.351145
50-60,0.141844,0.141844,0.141844,0.141844,0.2,0.141844,0.141844,0.141844,0.142349,0.141844,0.141844,0.142349,0.141844,0.142349,0.142349,0.142349,0.141844,0.143382,0.142349,0.142349,0.141844,0.141844,0.141844,0.141844,0.142349,0.142349,0.141844,0.141844,0.142349,0.142349,0.141844,0.144404,0.142349,0.143885,0.142349,0.141844,0.142349,0.142857,0.143382,0.142857,0.141844,0.142857,0.143369,0.142349,0.141844,0.141844,0.142349,0.142349,0.141844,0.142349,0.142857,0.141844,0.143382,0.141844,0.152091,0.142349,0.142349,0.142349,0.141844,0.148855
60+,0.102837,0.102837,0.102837,0.102837,0.4,0.102837,0.102837,0.102837,0.103203,0.102837,0.102837,0.103203,0.102837,0.099644,0.099644,0.099644,0.102837,0.095588,0.099644,0.099644,0.102837,0.102837,0.102837,0.102837,0.099644,0.099644,0.102837,0.102837,0.099644,0.099644,0.102837,0.101083,0.099644,0.100719,0.099644,0.102837,0.099644,0.098901,0.095588,0.100000,0.102837,0.100000,0.100358,0.099644,0.102837,0.102837,0.099644,0.099644,0.102837,0.099644,0.098901,0.102837,0.095588,0.102837,0.102662,0.099644,0.099644,0.099644,0.102837,0.099237


In [472]:
df_final.groupby('homeOwner').count()/df_final.groupby('homeOwner').count().sum()

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,amount,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,region,payDate,age,age_group,income_group,platform
homeOwner,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
own,0.315603,0.315603,0.315603,0.315603,0.0,0.315603,0.315603,0.315603,0.316726,0.315603,0.315603,0.316726,0.315603,0.316726,0.316726,0.316726,0.315603,0.327206,0.316726,0.316726,0.315603,0.315603,0.315603,0.315603,0.316726,0.316726,0.315603,0.315603,0.316726,0.316726,0.315603,0.3213,0.316726,0.320144,0.316726,0.315603,0.316726,0.326007,0.327206,0.317857,0.315603,0.317857,0.318996,0.316726,0.315603,0.315603,0.316726,0.316726,0.316726,0.326007,0.315603,0.327206,0.315603,0.334601,0.316726,0.316726,0.316726,0.315603,0.315603,0.339695
rent,0.684397,0.684397,0.684397,0.684397,1.0,0.684397,0.684397,0.684397,0.683274,0.684397,0.684397,0.683274,0.684397,0.683274,0.683274,0.683274,0.684397,0.672794,0.683274,0.683274,0.684397,0.684397,0.684397,0.684397,0.683274,0.683274,0.684397,0.684397,0.683274,0.683274,0.684397,0.6787,0.683274,0.679856,0.683274,0.684397,0.683274,0.673993,0.672794,0.682143,0.684397,0.682143,0.681004,0.683274,0.684397,0.684397,0.683274,0.683274,0.683274,0.673993,0.684397,0.672794,0.684397,0.665399,0.683274,0.683274,0.683274,0.684397,0.684397,0.660305


In [474]:
df_selected_email.groupby('homeOwner').count()/df_selected_email.groupby('homeOwner').count().sum()

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,amount,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,region,payDate,age,age_group,income_group,platform
homeOwner,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
own,0.37931,0.37931,0.37931,0.37931,NaN,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931,0.37931
rent,0.62069,0.62069,0.62069,0.62069,NaN,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069,0.62069


In [476]:
df_final.groupby('income_group').count()/df_final.groupby('income_group').count().sum() 

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,amount,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,homeOwner,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,region,payDate,age,age_group,platform
income_group,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3000 and under,0.003546,0.003546,0.003546,0.003546,0.0,0.003546,0.003546,0.003546,0.003559,0.003546,0.003546,0.003559,0.003546,0.003559,0.003559,0.003559,0.003546,0.000000,0.003559,0.003559,0.003546,0.003546,0.003546,0.003546,0.003559,0.003559,0.003546,0.003546,0.003559,0.003559,0.003546,0.000000,0.003559,0.000000,0.003559,0.003546,0.003559,0.000000,0.000000,0.003571,0.003546,0.003571,0.003584,0.003559,0.003546,0.003546,0.003559,0.003559,0.003546,0.003559,0.000000,0.003546,0.000000,0.003546,0.003802,0.003559,0.003559,0.003559,0.003546,0.000000
3000-4000,0.351064,0.351064,0.351064,0.351064,0.6,0.351064,0.351064,0.351064,0.352313,0.351064,0.351064,0.352313,0.351064,0.352313,0.352313,0.352313,0.351064,0.352941,0.352313,0.352313,0.351064,0.351064,0.351064,0.351064,0.352313,0.352313,0.351064,0.351064,0.352313,0.352313,0.351064,0.357401,0.352313,0.356115,0.352313,0.351064,0.352313,0.351648,0.352941,0.353571,0.351064,0.353571,0.354839,0.352313,0.351064,0.351064,0.352313,0.352313,0.351064,0.352313,0.351648,0.351064,0.352941,0.351064,0.365019,0.352313,0.352313,0.352313,0.351064,0.366412
4000-5000,0.117021,0.117021,0.117021,0.117021,0.0,0.117021,0.117021,0.117021,0.117438,0.117021,0.117021,0.117438,0.117021,0.117438,0.117438,0.117438,0.117021,0.121324,0.117438,0.117438,0.117021,0.117021,0.117021,0.117021,0.117438,0.117438,0.117021,0.117021,0.117438,0.117438,0.117021,0.119134,0.117438,0.118705,0.117438,0.117021,0.117438,0.120879,0.121324,0.114286,0.117021,0.114286,0.114695,0.117438,0.117021,0.117021,0.117438,0.117438,0.117021,0.117438,0.120879,0.117021,0.121324,0.117021,0.095057,0.117438,0.117438,0.117438,0.117021,0.099237
5000-6000,0.067376,0.067376,0.067376,0.067376,0.2,0.067376,0.067376,0.067376,0.067616,0.067376,0.067376,0.067616,0.067376,0.067616,0.067616,0.067616,0.067376,0.066176,0.067616,0.067616,0.067376,0.067376,0.067376,0.067376,0.067616,0.067616,0.067376,0.067376,0.067616,0.067616,0.067376,0.068592,0.067616,0.068345,0.067616,0.067376,0.067616,0.065934,0.066176,0.067857,0.067376,0.067857,0.068100,0.067616,0.067376,0.067376,0.067616,0.067616,0.067376,0.067616,0.065934,0.067376,0.066176,0.067376,0.072243,0.067616,0.067616,0.067616,0.067376,0.068702
6000+,0.460993,0.460993,0.460993,0.460993,0.2,0.460993,0.460993,0.460993,0.459075,0.460993,0.460993,0.459075,0.460993,0.459075,0.459075,0.459075,0.460993,0.459559,0.459075,0.459075,0.460993,0.460993,0.460993,0.460993,0.459075,0.459075,0.460993,0.460993,0.459075,0.459075,0.460993,0.454874,0.459075,0.456835,0.459075,0.460993,0.459075,0.461538,0.459559,0.460714,0.460993,0.460714,0.458781,0.459075,0.460993,0.460993,0.459075,0.459075,0.460993,0.459075,0.461538,0.460993,0.459559,0.460993,0.463878,0.459075,0.459075,0.459075,0.460993,0.465649


In [478]:
df_selected_email.groupby('income_group').count()/df_selected_email.groupby('income_group').count().sum()

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,amount,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,homeOwner,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,region,payDate,age,age_group,platform
income_group,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3000-4000,0.327586,0.327586,0.327586,0.327586,NaN,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586,0.327586
4000-5000,0.137931,0.137931,0.137931,0.137931,NaN,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931,0.137931
5000-6000,0.034483,0.034483,0.034483,0.034483,NaN,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483
6000+,0.500000,0.500000,0.500000,0.500000,NaN,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000


In [480]:
df_final.groupby('platform').count()/df_final.groupby('platform').count().sum()

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,amount,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,homeOwner,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,region,payDate,age,age_group,income_group
platform,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Linux,0.236641,0.236641,0.236641,0.236641,NaN,0.236641,0.236641,0.236641,0.237548,0.236641,0.236641,0.237548,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.236641,0.240157,0.236641,0.236641,0.236641,0.236641,0.236641
Macintosh,0.026718,0.026718,0.026718,0.026718,NaN,0.026718,0.026718,0.026718,0.026820,0.026718,0.026718,0.026820,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.026718,0.027559,0.026718,0.026718,0.026718,0.026718,0.026718
Windows NT 10.0,0.114504,0.114504,0.114504,0.114504,NaN,0.114504,0.114504,0.114504,0.114943,0.114504,0.114504,0.114943,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.114504,0.110236,0.114504,0.114504,0.114504,0.114504,0.114504
iPhone,0.622137,0.622137,0.622137,0.622137,NaN,0.622137,0.622137,0.622137,0.620690,0.622137,0.622137,0.620690,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622137,0.622047,0.622137,0.622137,0.622137,0.622137,0.622137


In [482]:
df_selected_email.groupby('platform').count()/df_selected_email.groupby('platform').count().sum()

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,amount,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,homeOwner,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,region,payDate,age,age_group,income_group
platform,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Linux,0.275862,0.275862,0.275862,0.275862,NaN,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862,0.275862
Macintosh,0.034483,0.034483,0.034483,0.034483,NaN,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483,0.034483
Windows NT 10.0,0.189655,0.189655,0.189655,0.189655,NaN,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655,0.189655
iPhone,0.500000,0.500000,0.500000,0.500000,NaN,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000


In [492]:
df_selected_email.groupby('amount').count()/df_selected_email.groupby('amount').count().sum()

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,homeOwner,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,region,payDate,age,age_group,income_group,platform
amount,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
200.0,0.206897,0.206897,0.206897,0.206897,NaN,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897,0.206897
300.0,0.155172,0.155172,0.155172,0.155172,NaN,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172,0.155172
400.0,0.120690,0.120690,0.120690,0.120690,NaN,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690
450.0,0.017241,0.017241,0.017241,0.017241,NaN,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241
500.0,0.310345,0.310345,0.310345,0.310345,NaN,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345,0.310345
600.0,0.120690,0.120690,0.120690,0.120690,NaN,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690
700.0,0.034483,0.034483,0.034483,0.034483,NaN,0.034483,0.034483,0.034483,0.034483,0

In [484]:
 df_final.groupby('region').count()/df_final.groupby('region').count().sum()

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,amount,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,homeOwner,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,payDate,age,age_group,income_group,platform
region,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
CA,0.213523,0.213523,0.213523,0.213523,0.2,0.213523,0.213523,0.213523,0.214286,0.213523,0.213523,0.214286,0.213523,0.213523,0.213523,0.213523,0.213523,0.202206,0.213523,0.213523,0.213523,0.213523,0.213523,0.213523,0.213523,0.213523,0.213523,0.213523,0.213523,0.213523,0.213523,0.202166,0.213523,0.205036,0.213523,0.213523,0.213523,0.202206,0.202206,0.210714,0.213523,0.210714,0.207885,0.213523,0.213523,0.213523,0.213523,0.213523,0.213523,0.213523,0.202206,0.213523,0.202206,0.213523,0.190114,0.213523,0.213523,0.213523,0.213523,0.171756
DE,0.007117,0.007117,0.007117,0.007117,0.0,0.007117,0.007117,0.007117,0.007143,0.007117,0.007117,0.007143,0.007117,0.007117,0.007117,0.007117,0.007117,0.007353,0.007117,0.007117,0.007117,0.007117,0.007117,0.007117,0.007117,0.007117,0.007117,0.007117,0.007117,0.007117,0.007117,0.007220,0.007117,0.007194,0.007117,0.007117,0.007117,0.007353,0.007353,0.007143,0.007117,0.007143,0.007168,0.007117,0.007117,0.007117,0.007117,0.007117,0.007117,0.007117,0.007353,0.007117,0.007353,0.007117,0.007605,0.007117,0.007117,0.007117,0.007117,0.007634
FL,0.085409,0.085409,0.085409,0.085409,0.0,0.085409,0.085409,0.085409,0.085714,0.085409,0.085409,0.085714,0.085409,0.085409,0.085409,0.085409,0.085409,0.088235,0.085409,0.085409,0.085409,0.085409,0.085409,0.085409,0.085409,0.085409,0.085409,0.085409,0.085409,0.085409,0.085409,0.086643,0.085409,0.086331,0.085409,0.085409,0.085409,0.088235,0.088235,0.085714,0.085409,0.085714,0.086022,0.085409,0.085409,0.085409,0.085409,0.085409,0.085409,0.085409,0.088235,0.085409,0.088235,0.085409,0.087452,0.085409,0.085409,0.085409,0.085409,0.091603
HI,0.003559,0.003559,0.003559,0.003559,0.0,0.003559,0.003559,0.003559,0.003571,0.003559,0.003559,0.003571,0.003559,0.003559,0.003559,0.003559,0.003559,0.003676,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003610,0.003559,0.003597,0.003559,0.003559,0.003559,0.003676,0.003676,0.003571,0.003559,0.003571,0.003584,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003676,0.003559,0.003676,0.003559,0.003802,0.003559,0.003559,0.003559,0.003559,0.003817
IA,0.010676,0.010676,0.010676,0.010676,0.0,0.010676,0.010676,0.010676,0.010714,0.010676,0.010676,0.010714,0.010676,0.010676,0.010676,0.010676,0.010676,0.011029,0.010676,0.010676,0.010676,0.010676,0.010676,0.010676,0.010676,0.010676,0.010676,0.010676,0.010676,0.010676,0.010676,0.010830,0.010676,0.010791,0.010676,0.010676,0.010676,0.011029,0.011029,0.010714,0.010676,0.010714,0.010753,0.010676,0.010676,0.010676,0.010676,0.010676,0.010676,0.010676,0.011029,0.010676,0.011029,0.010676,0.011407,0.010676,0.010676,0.010676,0.010676,0.011450
ID,0.003559,0.003559,0.003559,0.003559,0.0,0.003559,0.003559,0.003559,0.003571,0.003559,0.003559,0.003571,0.003559,0.003559,0.003559,0.003559,0.003559,0.003676,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003610,0.003559,0.003597,0.003559,0.003559,0.003559,0.003676,0.003676,0.003571,0.003559,0.003571,0.003584,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003559,0.003676,0.003559,0.003676,0.003559,0.003802,0.003559,0.003559,0.003559,0.003559,0.003817
IN,0.017794,0.017794,0.017794,0.017794,0.0,0.017794,0.017794,0.017794,0.017857,0.017794,0.017794,0.01

In [486]:
df_selected_email.groupby('region').count()/df_selected_email.groupby('region').count().sum()

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,amount,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,homeOwner,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,payDate,age,age_group,income_group,platform
region,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
CA,0.172414,0.172414,0.172414,0.172414,NaN,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414,0.172414
FL,0.120690,0.120690,0.120690,0.120690,NaN,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690,0.120690
HI,0.017241,0.017241,0.017241,0.017241,NaN,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241
IN,0.017241,0.017241,0.017241,0.017241,NaN,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241
LA,0.017241,0.017241,0.017241,0.017241,NaN,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241
MI,0.051724,0.051724,0.051724,0.051724,NaN,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724,0.051724
MN,0.017241,0.017241,0.017241,0.017241,NaN,0.017241,0.017241,0.017241,0.017241,0.017241,0.017241,0.01

In [488]:
df_selected_email[df_selected_email['region'] == 'FL']

,id,ping_id,jluvr,id_ad,id_brand,id_partner,partner_name,ping_group,raw_response_json,accepted,request_datetime,response_datetime,last_modified,income,lastName,driverState,purpose,postalCode,jobTitle,bankName,bankruptcy,mailing,directDeposit,frequency,driverLicense,company,military,accountLength,email,timeJob,aff,amount,creditScore,payDateNext,homePhone,accountType,locality,ipAddress,userAgent,accountNumber,timeAddress,accountABA,SSN,firstName,apiToken,termsAgreed,mobilePhone,streetAddress,homeOwner,dob,domain,minPrice,workPhone,incomeSource,timeAllowed,region,payDate,age,age_group,income_group,platform
51,7133,e19c06b0-2062-4d86-a811-2ee9f2a0b710,e8bb9259-4aaf-4cce-b551-056241fcbbe6,CSRC941018000,NaN,7,stopGoNetwork,personal_loan,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<respo...",1,28/7/2025 15:48:20,28/7/2025 15:48:31,28/7/2025 15:48:31,3400.0,Martinez,FL,other,33935,Customer servixe,CHASE BANK,N,Y,Y,biWeekly,M635533735500,Echo sphere,N,36,lcmartinezz73@gmail.com,21,NIC001XP,500.0,poor,2025-08-22,8633739573,checking,Labelle,71.200.201.138,Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like M...,990764331,36,267084131,sQlX1sTmNBnbOarXufLG7A==,Lisa,1cc292a5-2626-45dc-88be-2cb062c6fe2b,Y,8633739573,"353, BELMONT ST",rent,1973-02-10,https://web.creditscorereportcard.com/lead?jlu...,1.0,8633739573,employed,180,FL,2025-08-08,52.0,50-60,3000-4000,iPhone
110,6996,ddb1d5ef-0631-4c76-bb20-bc8a4ab870c5,f9f9efdc-30fe-474a-9084-1afdf0461989,CSRC941018000,NaN,7,stopGoNetwork,personal_loan,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<respo...",1,24/7/2025 15:31:13,24/7/2025 15:31:22,24/7/2025 15:31:22,4750.0,Stock,FL,other,34772,Practice supervisor,USAA FEDERAL SAVINGS BANK,N,Y,Y,biWeekly,S320433917260,Orlando Health,N,3,jordan.rambo@gmail.com,219,NIC001XP,500.0,poor,2025-08-22,4073616339,checking,Saint Cloud,104.28.94.199,Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like M...,0346048702,85,314074269,3faQ7iUTAwmwKLJXO2esqg==,Jordan,1cc292a5-2626-45dc-88be-2cb062c6fe2b,Y,4073616339,"4475, LASALLE AVENUE SOUTH",own,1991-06-26,https://web.creditscorereportcard.com/lead?jlu...,1.0,4078737908,employed,180,FL,2025-08-08,34.0,30-40,4000-5000,iPhone
134,6962,519b834c-b850-418c-8576-0df77d829416,85c038ee-6f70-4d22-8d0d-c2a0bd21bd74,CSRC941018000,NaN,7,stopGoNetwork,personal_loan,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<respo...",1,23/7/2025 15:22:25,23/7/2025 15:24:37,23/7/2025 15:24:37,5000.0,Jackson,FL,other,32808,Manager,ADDITION FINANCIAL,N,Y,Y,biWeekly,J250104809180,AdCoQuantum,N,36,cammbae41@gmail.com,146,NIC001XP,300.0,good,2025-08-22,3212081591,checking,Orlando,71.44.156.44,Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like M...,0000701366,36,263181384,FhAFfBLHTWSVhDuGpceslg==,Camelia,1cc292a5-2626-45dc-88be-2cb062c6fe2b,Y,3212081591,"5426, AEOLUS WAY",rent,1980-11-18,https://web.creditscorereportcard.com/lead?jlu...,1.0,4072965842,employed,180,FL,2025-08-08,44.0,40-50,6000+,iPhone
153,6934,89f35fbb-04c6-4b92-bdde-3aefbc3a7eff,dc381c6d-c0aa-42f3-9f00-e9473bf89dbd,CSRC941018000,NaN,7,stopGoNetwork,personal_loan,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<respo...",1,22/7/2025 15:51:47,22/7/2025 15:51:55,22/7/2025 15:51:55,3200.0,Ward,FL,other,33881,Front end tech 1,MIDFLORIDA CREDIT UNION,N,Y,Y,biWeekly,W630012807932,Cvs specialty,N,85,gatormomma30@gmail.com,18,NIC001XP,200.0,poor,2025-08-08,8635212830,checking,Winter Haven,47.204.174.147,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1423345885,85,263179804,p4+rFkb1/2h3AhAhSiGSqg==,Amanda,1cc292a5-2626-45dc-88be-2cb062c6fe2b,Y,8635212830,"1700, 15TH STREET NORTHWEST",own,1980-08-13,https://web.creditscorereportcard.com/lead?jlu...,1.0,8002372767,employed,180,FL,2025-07-25,44.0,40-50,3000-4000,Windows NT 10.0
174,6908,b6b46e7b-5e6a-48e9-b192-d64b7f86c1c9,7a412ceb-2256-4fd3-88c4-7f9b46c20be5,CSRC941018000,NaN,7,stopGoNetwork,personal_loan,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<respo...",1,21/7/2025 17:30:30,21/7/2025 17:30:37,21/7/2025 17:30:37,3800.0,Cucura,FL,other,33409,Battery/waitre

In [490]:
# hfaui fhaie fer tfi iwery9 pqeirh guweaigiueg 7auie thfnhjkdbv auy fhuaf laf lwefuu lufhj gfhwae fle,af wyufj  fuwal fauhj